In [3]:


# =====================================================================
# AGENTIC FINANCEBENCH — Llama 3.1 8B + RAG + Agent Tools
# =====================================================================

# =====================================================================
# CELL 1: Install Dependencies
# =====================================================================

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('✅ Cell 1 done — Dependencies installed')

✅ Cell 1 done — Dependencies installed


In [4]:
# =====================================================================
# CELL 2: Imports
# =====================================================================

import os, re, json, torch, gc
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

os.makedirs('results', exist_ok=True)
print('✅ Cell 2 done — Imports ready')

✅ Cell 2 done — Imports ready


In [5]:
# =====================================================================
# CELL 3: Load FinanceBench Data
# =====================================================================

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

qa = load_jsonl('financebench_open_source.jsonl')

evidence_map = {}
for item in qa:
    evidence_map[item['financebench_id']] = item.get('evidence', [])

print(f'✅ Cell 3 done — Loaded {len(qa)} questions')

✅ Cell 3 done — Loaded 150 questions


In [6]:
# =====================================================================
# CELL 4: Load Llama 3.1 8B
# =====================================================================

from transformers import AutoTokenizer, AutoModelForCausalLM

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Memory: {mem_gb:.1f} GB')

MODEL_NAME = 'meta-llama/Llama-3.1-8B-Instruct'
print(f'\nLoading {MODEL_NAME} in float16...')

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()

# Cast to bfloat16
for name, param in model.named_parameters():
    if param.dtype != torch.bfloat16:
        param.data = param.data.to(torch.bfloat16)
for name, buf in model.named_buffers():
    if buf.dtype.is_floating_point and buf.dtype != torch.bfloat16:
        buf.data = buf.data.to(torch.bfloat16)

mem_used = torch.cuda.memory_allocated() / 1e9
print(f'✅ Model loaded! GPU memory: {mem_used:.1f} GB')

CUDA: True
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB

Loading meta-llama/Llama-3.1-8B-Instruct in float16...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

✅ Model loaded! GPU memory: 16.1 GB


In [7]:
# =====================================================================
# CELL 5: Core Helper Functions (unchanged from original)
# =====================================================================

def extract_all_numbers(text):
    if not text:
        return []
    cleaned = text.replace('$', '').replace('%', '')
    numbers = []
    used_positions = set()
    for m in re.finditer(r'\((\d[\d,]*(?:\.\d+)?)\)', cleaned):
        num_str = m.group(1).replace(',', '')
        try:
            numbers.append(-float(num_str))
            for pos in range(m.start(), m.end()):
                used_positions.add(pos)
        except:
            pass
    for m in re.finditer(r'-?\d[\d,]*\.?\d*', cleaned):
        if any(pos in used_positions for pos in range(m.start(), m.end())):
            continue
        num_str = m.group(0).replace(',', '')
        try:
            numbers.append(float(num_str))
        except:
            pass
    return numbers

def extract_first_number(text):
    nums = extract_all_numbers(text)
    return nums[0] if nums else None

def normalize_value_to_base(value, text):
    text_lower = text.lower()
    if re.search(r'\bbillion\b', text_lower) or re.search(r'\d\s*b\b', text_lower):
        return value * 1_000_000_000
    if re.search(r'\bmillion\b', text_lower) or re.search(r'\d\s*m\b', text_lower):
        return value * 1_000_000
    if re.search(r'\bthousand\b', text_lower) or re.search(r'\d\s*k\b', text_lower):
        return value * 1_000
    return value

def numerical_accuracy(pred, gold):
    gold_nums = extract_all_numbers(gold)
    pred_nums = extract_all_numbers(pred)
    if not gold_nums:
        return 1.0 if not pred_nums else 0.0
    if not pred_nums:
        return 0.0
    for g in gold_nums:
        matched = False
        for p in pred_nums:
            if g == 0:
                if abs(p) < 1e-6: matched = True; break
            elif abs(p - g) / abs(g) < 0.05:
                matched = True; break
        if not matched:
            g_base = normalize_value_to_base(g, gold)
            for p in pred_nums:
                p_base = normalize_value_to_base(p, pred)
                if g_base != 0 and abs(p_base - g_base) / abs(g_base) < 0.05:
                    matched = True; break
            if not matched:
                return 0.0
    return 1.0

def numerical_hallucination_rate(pred, evidence_list):
    if not pred:
        return 0.0
    pred_nums = extract_all_numbers(pred)
    if not pred_nums:
        return 0.0
    evidence_nums = set()
    for ev in evidence_list:
        if isinstance(ev, dict):
            ev_text = ev.get('evidence_text', '') + ' ' + ev.get('evidence_text_full_page', '')
        else:
            ev_text = str(ev)
        for n in extract_all_numbers(ev_text):
            evidence_nums.add(abs(n))
    if not evidence_nums:
        return 1.0
    hallucinated = 0
    for p in pred_nums:
        p_abs = abs(p)
        found = False
        for e in evidence_nums:
            if e == 0:
                if p_abs < 1e-6: found = True; break
            elif abs(p_abs - e) / max(abs(e), 1e-10) < 0.05:
                found = True; break
        if not found:
            hallucinated += 1
    return hallucinated / len(pred_nums)

def calculate_bleu(reference, candidate):
    if not reference or not candidate:
        return 0.0
    ref_tokens = [reference.lower().split()]
    cand_tokens = candidate.lower().split()
    if not cand_tokens:
        return 0.0
    smoothing = SmoothingFunction().method1
    try:
        return sentence_bleu(ref_tokens, cand_tokens, smoothing_function=smoothing)
    except:
        return 0.0

def has_citation(candidate):
    if not candidate:
        return 0.0
    candidate_lower = candidate.lower()
    patterns = [
        r'according to', r'based on',
        r'as (stated|reported|shown|noted|mentioned) in',
        r'per the', r'page \d+',
        r'10-k\b', r'10k\b', r'10-q\b',
        r'\bfiling\b', r'\bannual report\b',
        r'financial statement', r'cash flow statement',
        r'income statement', r'balance sheet',
        r'\bsec\b', r'\b(form|item)\s+\d',
        r'source:', r'\[.*?\]', r'evidence',
    ]
    return 1.0 if any(re.search(p, candidate_lower) for p in patterns) else 0.0

def normalize(s):
    return re.sub(r'\s+', ' ', s.strip().lower())

def is_correct(pred, gold):
    if normalize(pred) == normalize(gold):
        return True
    p = extract_first_number(pred)
    g = extract_first_number(gold)
    if p is not None and g is not None:
        if g != 0 and abs(p - g) / abs(g) < 0.05:
            return True
        p_base = normalize_value_to_base(p, pred)
        g_base = normalize_value_to_base(g, gold)
        if g_base != 0 and abs(p_base - g_base) / abs(g_base) < 0.05:
            return True
    return False

def extract_answer_from_verbose(pred):
    if not pred:
        return pred
    dollar = re.search(r'\$\s*[\d,]+(?:\.\d+)?\s*(?:billion|million|B|M|K)?', pred, re.IGNORECASE)
    if dollar:
        return dollar.group(0).strip()
    pct = re.search(r'[-+]?[\d,]+(?:\.\d+)?\s*%', pred)
    if pct:
        return pct.group(0).strip()
    with_units = re.search(r'[-+]?[\d,]+(?:\.\d+)?\s*(?:billion|million|thousand|B|M|K)', pred, re.IGNORECASE)
    if with_units:
        return with_units.group(0).strip()
    plain = re.search(r'[-+]?\$?\s*[\d,]+(?:\.\d+)?', pred)
    if plain:
        return plain.group(0).strip()
    if len(pred.split()) < 15:
        return pred
    return pred

print('Cell 5 done — Helper functions ready')


✅ Cell 5 done — Helper functions ready


In [8]:
# =====================================================================
# CELL 6: LLM Generation Function
# =====================================================================

def generate_answer(prompt, max_new_tokens=128):
    messages = [{"role": "user", "content": prompt}]
    input_text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(input_text, return_tensors='pt', truncation=True, max_length=3000)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    raw = tok.decode(new_tokens, skip_special_tokens=True).strip()
    return raw

test = generate_answer('What is 2+2? Just the number.')
print(f' Generation test: "{test[:50]}"')

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Generation test: "4"


In [10]:
!pip install -q rank_bm25
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 113.1 MB/s eta 0:00:00


In [11]:
# =====================================================================
# CELL 7: RAG System (unchanged from original)
# =====================================================================

from rank_bm25 import BM25Okapi
import pickle, shutil
import faiss
from sentence_transformers import SentenceTransformer

os.makedirs('rag/vector_index', exist_ok=True)

if not os.path.exists('rag/vector_index/faiss_index.bin'):
    print('Upload faiss_index.bin and evidence_store.pkl:')
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        if 'faiss' in fname:
            with open('rag/vector_index/faiss_index.bin', 'wb') as f:
                f.write(uploaded[fname])
        if 'evidence' in fname or 'pkl' in fname:
            with open('rag/vector_index/evidence_store.pkl', 'wb') as f:
                f.write(uploaded[fname])
else:
    print('RAG files already exist')

COMPANY_ALIASES = {
    '3m': '3M', 'mmm': '3M',
    'aes': 'AES Corporation', 'aes corporation': 'AES Corporation',
    'amcor': 'Amcor', 'amd': 'AMD', 'advanced micro': 'AMD',
    'activision': 'Activision Blizzard', 'activision blizzard': 'Activision Blizzard',
    'adobe': 'Adobe',
    'american express': 'American Express', 'amex': 'American Express',
    'american water': 'American Water Works', 'american water works': 'American Water Works',
    'best buy': 'Best Buy', 'boeing': 'Boeing',
    'cvs': 'CVS Health', 'cvs health': 'CVS Health',
    'coca-cola': 'Coca-Cola', 'coca cola': 'Coca-Cola', 'coke': 'Coca-Cola',
    'corning': 'Corning', 'costco': 'Costco', 'foot locker': 'Foot Locker',
    'general mills': 'General Mills',
    'jpmorgan': 'JPMorgan', 'jpm': 'JPMorgan', 'jp morgan': 'JPMorgan',
    'johnson': 'Johnson & Johnson', 'johnson & johnson': 'Johnson & Johnson', 'j&j': 'Johnson & Johnson',
    'lockheed': 'Lockheed Martin', 'lockheed martin': 'Lockheed Martin',
    'mgm': 'MGM Resorts', 'mgm resorts': 'MGM Resorts',
    'microsoft': 'Microsoft', 'msft': 'Microsoft',
    'nike': 'Nike', 'netflix': 'Netflix', 'nflx': 'Netflix',
    'paypal': 'Paypal', 'pepsico': 'PepsiCo', 'pepsi': 'PepsiCo',
    'pfizer': 'Pfizer', 'ulta': 'Ulta Beauty', 'ulta beauty': 'Ulta Beauty',
    'verizon': 'Verizon', 'walmart': 'Walmart',
    'block': 'Block', 'square': 'Block',
    'amazon': 'Amazon', 'amzn': 'Amazon',
    'kraft': 'Kraft Heinz', 'kraft heinz': 'Kraft Heinz',
}

def extract_company(question):
    q_lower = question.lower()
    for alias in sorted(COMPANY_ALIASES.keys(), key=len, reverse=True):
        if alias in q_lower:
            return COMPANY_ALIASES[alias]
    return None

def extract_year(question):
    fy_match = re.search(r'FY\s*(\d{4})', question, re.IGNORECASE)
    if fy_match:
        return fy_match.group(1)
    year_match = re.search(r'20\d{2}', question)
    if year_match:
        return year_match.group(0)
    return None

STATEMENT_KEYWORDS = {
    'cash_flow': [
        'capital expenditure', 'capex', 'cash flow', 'cash provided',
        'purchases of property', 'dividends paid', 'free cash flow',
        'depreciation', 'amortization', 'investing activities',
        'financing activities', 'operating activities', 'fcf',
        'cash used', 'cash generated', 'repurchase',
    ],
    'income': [
        'revenue', 'sales', 'net income', 'operating income',
        'gross profit', 'earnings', 'eps', 'earnings per share',
        'operating margin', 'gross margin', 'profit margin',
        'cost of goods', 'cost of revenue', 'operating expense',
        'ebitda', 'ebit', 'income tax', 'net profit',
        'top line', 'bottom line', 'r&d', 'research and development',
        'sg&a', 'selling general',
    ],
    'balance': [
        'total assets', 'total liabilities', 'current assets',
        'current liabilities', 'equity', 'stockholders equity',
        'long-term debt', 'short-term debt', 'book value',
        'working capital', 'current ratio', 'quick ratio',
        'inventory', 'accounts receivable', 'accounts payable',
        'goodwill', 'intangible', 'net assets', 'ppe',
        'property plant', 'fixed asset',
    ],
}

STATEMENT_TEXT_MARKERS = {
    'cash_flow': [
        'cash flows from', 'cash provided by', 'investing activities',
        'financing activities', 'statement of cash flow', 'cash used in',
        'purchases of property',
    ],
    'income': [
        'net revenues', 'net sales', 'total revenues', 'cost of',
        'operating income', 'statements of operations',
        'statements of income', 'gross profit',
    ],
    'balance': [
        'total assets', 'total liabilities', 'stockholders equity',
        'balance sheet', 'current assets', 'current liabilities',
    ],
}

def detect_statement_type(question):
    q_lower = question.lower()
    scores = {'cash_flow': 0, 'income': 0, 'balance': 0}
    for stmt_type, keywords in STATEMENT_KEYWORDS.items():
        for kw in keywords:
            if kw in q_lower:
                scores[stmt_type] += 1
    best = max(scores, key=scores.get)
    return best if scores[best] > 0 else None

def score_statement_match(evidence_text, target_statement):
    if not target_statement:
        return 0
    text_lower = evidence_text.lower()
    markers = STATEMENT_TEXT_MARKERS.get(target_statement, [])
    return sum(1 for m in markers if m in text_lower)

class FinancialRAG:
    def __init__(self, embedding_model='all-MiniLM-L6-v2'):
        print(f"Loading embedding model: {embedding_model}")
        self.encoder = SentenceTransformer(embedding_model)
        self.evidence_store = []
        self.index = None
        self.all_embeddings = None

    def load_index(self, index_dir='./rag/vector_index'):
        self.index = faiss.read_index(f'{index_dir}/faiss_index.bin')
        with open(f'{index_dir}/evidence_store.pkl', 'rb') as f:
            self.evidence_store = pickle.load(f)
        self.all_embeddings = np.zeros((self.index.ntotal, self.index.d), dtype='float32')
        for i in range(self.index.ntotal):
            self.all_embeddings[i] = self.index.reconstruct(i)
        print(f"Index loaded: {self.index.ntotal} vectors, {len(self.evidence_store)} chunks")

    def _cosine_sim(self, query_emb, doc_embs):
        query_norm = query_emb / (np.linalg.norm(query_emb) + 1e-10)
        doc_norms = doc_embs / (np.linalg.norm(doc_embs, axis=1, keepdims=True) + 1e-10)
        return np.dot(doc_norms, query_norm.T).flatten()

    def retrieve(self, question, k=3, override_statement=None):
        question_embedding = self.encoder.encode([question], convert_to_numpy=True).astype('float32')
        target_company = extract_company(question)
        target_year = extract_year(question)
        target_statement = override_statement if override_statement else detect_statement_type(question)

        filtered = []
        for i, (text, metadata) in enumerate(self.evidence_store):
            meta_company = metadata.get('company', '').lower()
            meta_doc = metadata.get('doc_name', '')
            company_match = target_company and target_company.lower() in meta_company
            year_match = target_year and target_year in meta_doc
            if company_match and year_match:
                filtered.append(i)

        if len(filtered) == 0:
            for i, (text, metadata) in enumerate(self.evidence_store):
                meta_company = metadata.get('company', '').lower()
                if target_company and target_company.lower() in meta_company:
                    filtered.append(i)

        if len(filtered) == 0:
            filtered = list(range(len(self.evidence_store)))

        filtered_embeddings = self.all_embeddings[filtered]
        cos_scores = self._cosine_sim(question_embedding[0], filtered_embeddings)

        if cos_scores.max() != cos_scores.min():
            cos_norm = (cos_scores - cos_scores.min()) / (cos_scores.max() - cos_scores.min())
        else:
            cos_norm = np.ones_like(cos_scores)

        filtered_texts = [self.evidence_store[i][0] for i in filtered]
        tokenized_docs = [doc.lower().split() for doc in filtered_texts]
        bm25 = BM25Okapi(tokenized_docs)
        bm25_scores = bm25.get_scores(question.lower().split())

        if bm25_scores.max() != bm25_scores.min():
            bm25_norm = (bm25_scores - bm25_scores.min()) / (bm25_scores.max() - bm25_scores.min())
        else:
            bm25_norm = np.zeros_like(bm25_scores)

        stmt_scores = np.zeros(len(filtered))
        if target_statement:
            for j, idx in enumerate(filtered):
                text = self.evidence_store[idx][0]
                stmt_scores[j] = score_statement_match(text, target_statement)
            if stmt_scores.max() > 0:
                stmt_norm = stmt_scores / stmt_scores.max()
            else:
                stmt_norm = np.zeros_like(stmt_scores)
        else:
            stmt_norm = np.zeros_like(stmt_scores)

        combined_scores = 0.4 * cos_norm + 0.3 * bm25_norm + 0.3 * stmt_norm
        actual_k = min(k, len(filtered))
        top_indices = np.argsort(combined_scores)[::-1][:actual_k]

        candidates = []
        for idx in top_indices:
            real_idx = filtered[idx]
            text, metadata = self.evidence_store[real_idx]
            candidates.append({
                'evidence_text': text,
                'metadata': metadata,
                'distance': float(1 - combined_scores[idx]),
            })
        return candidates

rag = FinancialRAG()
rag.load_index('./rag/vector_index')
print('Cell 7 done — RAG system ready')

Upload faiss_index.bin and evidence_store.pkl:


Saving faiss_index.bin to faiss_index.bin
Saving evidence_store.pkl to evidence_store.pkl
Loading embedding model: all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Index loaded: 189 vectors, 189 chunks
✅ Cell 7 done — RAG system ready


In [12]:
# =====================================================================
# CELL 8: AGENT TOOLS
# These are the tools the agent can call during reasoning
# =====================================================================

# ── Tool 1: Calculator ────────────────────────────────────────────────
# Safely evaluates math expressions the LLM constructs

def calculator_tool(expression: str) -> str:
    """
    Safely evaluate a math expression string.
    Examples: "6489 / 267", "(177866 - 135987) / 135987 * 100"
    Returns the numeric result as a string, or an error message.
    """
    try:
        # Only allow safe math characters
        safe_expr = re.sub(r'[^0-9\+\-\*\/\(\)\.\s]', '', expression)
        if not safe_expr.strip():
            return "ERROR: Empty expression after cleaning"
        result = eval(safe_expr)
        # Format nicely
        if isinstance(result, float):
            if abs(result) > 1000:
                return f"{result:,.2f}"
            else:
                return f"{result:.4f}"
        return str(result)
    except Exception as e:
        return f"ERROR: {e}"


# ── Tool 2: Retrieve Evidence (wraps existing RAG) ────────────────────

def retrieve_tool(question: str, statement_type: str = None, k: int = 5) -> str:
    """
    Retrieve relevant evidence chunks for a question.
    statement_type: 'income', 'balance', 'cash_flow', or None for auto-detect
    Returns formatted evidence text.
    """
    results = rag.retrieve(question, k=k, override_statement=statement_type)
    if not results:
        return "No evidence found."
    parts = []
    for i, r in enumerate(results):
        meta = r['metadata']
        parts.append(
            f"[Chunk {i+1} | {meta.get('company','?')} | {meta.get('doc_name','?')}]\n"
            f"{r['evidence_text'][:1500]}"
        )
    return "\n\n".join(parts)


# ── Tool 3: Extract Numbers from Evidence ────────────────────────────

def extract_numbers_tool(text: str, label: str = "") -> str:
    """
    Extract all numbers from a text chunk.
    Useful when agent needs to find specific values to plug into a formula.
    """
    nums = extract_all_numbers(text)
    if not nums:
        return f"No numbers found in: {label or text[:80]}"
    # Show unique numbers sorted
    unique_nums = sorted(set(nums), reverse=True)
    return f"Numbers found ({label}): {unique_nums[:20]}"


# ── Tool 4: Verify Answer Against Evidence ───────────────────────────

def verify_tool(answer: str, evidence: str) -> str:
    """
    Check if the numbers in the answer actually appear in the evidence.
    Returns 'VERIFIED', 'PARTIAL', or 'NOT FOUND' with details.
    """
    answer_nums = extract_all_numbers(answer)
    evidence_nums = extract_all_numbers(evidence)

    if not answer_nums:
        return "VERIFIED (no numbers to verify)"

    evidence_set = set(abs(n) for n in evidence_nums)
    verified = []
    not_found = []

    for a_num in answer_nums:
        a_abs = abs(a_num)
        found = any(
            abs(a_abs - e) / max(abs(e), 1e-10) < 0.05
            for e in evidence_set
        )
        if found:
            verified.append(a_num)
        else:
            not_found.append(a_num)

    if not not_found:
        return f"VERIFIED — all numbers {verified} found in evidence"
    elif verified:
        return f"PARTIAL — {verified} found, {not_found} NOT in evidence"
    else:
        return f"NOT FOUND — {not_found} not in evidence — answer may be hallucinated"


print('Cell 8 done — Agent tools ready')
print('   Tools: calculator_tool, retrieve_tool, extract_numbers_tool, verify_tool')


✅ Cell 8 done — Agent tools ready
   Tools: calculator_tool, retrieve_tool, extract_numbers_tool, verify_tool


In [13]:
# =====================================================================
# CELL 9: QUESTION CATEGORIZATION (same as original)
# =====================================================================

def categorize_question(gold_answer):
    gold_clean = gold_answer.strip().lower()
    if gold_clean.startswith('yes') or gold_clean.startswith('no'):
        return 'yes_no'
    words = gold_answer.strip().split()
    if len(words) <= 4:
        nums = extract_all_numbers(gold_answer)
        if nums:
            return 'numerical'
    return 'descriptive'

for ex in qa:
    ex['category'] = categorize_question(ex['answer'])

cat_counts = {}
for ex in qa:
    cat_counts[ex['category']] = cat_counts.get(ex['category'], 0) + 1

print(f'Question Categories:')
print(f'  Numerical:   {cat_counts.get("numerical", 0)}')
print(f'  Yes/No:      {cat_counts.get("yes_no", 0)}')
print(f'  Descriptive: {cat_counts.get("descriptive", 0)}')
print(f'  Total:       {len(qa)}')

Question Categories:
  Numerical:   53
  Yes/No:      37
  Descriptive: 60
  Total:       150


In [18]:
# =====================================================================
# CELL 10: AGENT PROMPT BUILDERS
# Each category gets a multi-step ReAct-style prompt
# =====================================================================

def build_agent_prompt_numerical(question, evidence_text):
    """
    For numerical questions:
    Agent is forced to: extract numbers → write formula → calculate → verify
    """
    return f"""You are a financial data extraction and calculation tool.
Read the EVIDENCE carefully and answer the QUESTION with ONLY numbers from the evidence.

EVIDENCE:
{evidence_text}

QUESTION: {question}

INSTRUCTIONS:
- ONLY use numbers that literally appear in the EVIDENCE text above
- DO NOT use numbers from memory or training data
- If the question asks for a single value (revenue, capex, etc.), find it directly — no formula needed
- If the question asks for a ratio or percentage change, write the formula and compute it
- End with exactly: FINAL ANSWER: [number with units]

STEP 1 - What value(s) does the question ask for?
Write: "Question asks for: [exact metric name]"

STEP 2 - Find the number(s) in the evidence.
Write: "From evidence: [metric] = [exact number as written in evidence]"
If two numbers needed: "From evidence: [A] = [number], [B] = [number]"

STEP 3 - Compute if needed (skip if value found directly).
Write: "Calculation: [A] / [B] = [result]" or "No calculation needed"

STEP 4 - Write final answer.
Write: "FINAL ANSWER: [value]"

Example for direct lookup:
Question asks for: capital expenditure FY2018
From evidence: capital expenditure = $1,577 million
No calculation needed
FINAL ANSWER: $1,577M

Example for ratio:
Question asks for: fixed asset turnover ratio
From evidence: revenue = $6,489M, net PP&E = $267M
Calculation: 6489 / 267 = 24.31
FINAL ANSWER: 24.31

Now answer the question:"""


def build_agent_prompt_yesno(question, evidence_text):
    """
    For yes/no questions:
    Agent forced to compute metric → compare to correct threshold → decide
    """
    return f"""You are a financial analyst. You must COMPUTE a metric from the evidence and THEN answer Yes or No.

EVIDENCE:
{evidence_text}

QUESTION: {question}

THRESHOLD REFERENCE (use these standard benchmarks):
- Quick Ratio / Current Ratio: healthy = ABOVE 1.0 (below 1.0 = NOT healthy)
- Capital Intensity Ratio (Assets/Revenue): intensive = ABOVE 1.0 (below 1.0 = NOT intensive)
- Debt/Equity: high leverage = ABOVE 2.0
- Operating Margin: improving = current year HIGHER than prior year
- Gross Margin: consistent = less than 2% change year over year
- Revenue growth: high growth = ABOVE 10% year over year

STEP 1 - METRIC: What metric must you compute to answer this question?
Write: "Need to compute: [metric name]"

STEP 2 - EXTRACT: Find the exact numbers from the evidence (only numbers in the evidence).
Write: "Numerator = [value from evidence]"
Write: "Denominator = [value from evidence]" (if ratio)

STEP 3 - CALCULATE: Do the arithmetic.
Write: "Calculation: [numerator] / [denominator] = [result]"
Or: "Calculation: ([new] - [old]) / [old] * 100 = [result]%"

STEP 4 - COMPARE: Compare your result to the threshold.
Write: "Result [X] is [above/below] threshold [Y], therefore [conclusion]"

STEP 5 - FINAL ANSWER:
Write: "FINAL ANSWER: Yes" or "FINAL ANSWER: No"
Then one sentence explanation with the computed value.

CRITICAL RULES:
- Quick ratio 0.96 → below 1.0 → NOT healthy → FINAL ANSWER: No
- Capital intensity 0.197 → below 1.0 → NOT intensive → FINAL ANSWER: No
- If the metric result clearly crosses the threshold in the negative direction, answer No

Now answer:"""


def build_agent_prompt_descriptive(question, evidence_text):
    """
    For descriptive questions:
    Forces prose answer, no bullet points, cite source explicitly
    """
    return f"""You are a financial analyst. Answer the question using ONLY the evidence provided.
Write in complete sentences. Do NOT use bullet points or dashes. Do NOT say "STEP 1" etc.

EVIDENCE:
{evidence_text}

QUESTION: {question}

Instructions:
- Read the evidence carefully
- Find the specific facts that answer the question
- Write a 2-3 sentence answer in plain English
- Include specific numbers from the evidence
- End with: "FINAL ANSWER: [your answer in 1-2 sentences with numbers and source citation]"

Example format:
FINAL ANSWER: According to the 3M_2022_10K income statement, operating margin declined by 1.7% primarily due to higher raw material costs of $X million and supply chain disruptions.

Now answer the question:"""


print(' Cell 10 done — Agent prompts ready')

✅ Cell 10 done — Agent prompts ready


In [19]:
# =====================================================================
# CELL 11: MULTI-HOP RETRIEVAL
# For questions that require data from multiple statements/years
# =====================================================================

def detect_multi_hop(question):
    """
    Detect if question needs data from multiple sources.
    Returns list of (statement_type, year_hint) tuples to retrieve.
    """
    q_lower = question.lower()
    retrieval_tasks = []

    # Multi-year comparison
    years = re.findall(r'20\d{2}', question)
    fy_years = re.findall(r'FY\s*(\d{4})', question, re.IGNORECASE)
    all_years = list(set(years + fy_years))

    # Ratio questions that need two statements
    needs_balance_and_income = any(kw in q_lower for kw in [
        'asset turnover', 'return on asset', 'return on equity',
        'roa', 'roe', 'fixed asset turnover'
    ])
    needs_balance_and_cashflow = any(kw in q_lower for kw in [
        'operating cash flow ratio', 'cash flow to debt',
        'fcf to revenue', 'free cash flow ratio'
    ])
    needs_multi_year = len(all_years) >= 2 or any(kw in q_lower for kw in [
        'year-over-year', 'yoy', 'change from', 'growth', '3 year average',
        'compare', 'increased', 'decreased', 'vs fy', 'to fy'
    ])

    if needs_balance_and_income:
        retrieval_tasks = [('balance', None), ('income', None)]
    elif needs_balance_and_cashflow:
        retrieval_tasks = [('balance', None), ('cash_flow', None)]
    elif needs_multi_year and len(all_years) >= 2:
        stmt = detect_statement_type(question)
        retrieval_tasks = [(stmt, yr) for yr in all_years[:2]]
    else:
        retrieval_tasks = [(detect_statement_type(question), None)]

    return retrieval_tasks


def multi_hop_retrieve(question, k_per_hop=3):
    """
    Retrieve evidence from multiple sources when needed.
    Returns combined evidence text with source labels.
    """
    tasks = detect_multi_hop(question)
    all_evidence = []
    seen_chunks = set()

    for stmt_type, year_hint in tasks:
        # Temporarily patch question if targeting specific year
        q_for_retrieval = question
        if year_hint:
            q_for_retrieval = f"{question} FY{year_hint}"

        results = rag.retrieve(q_for_retrieval, k=k_per_hop, override_statement=stmt_type)
        for r in results:
            chunk_key = r['evidence_text'][:100]
            if chunk_key not in seen_chunks:
                seen_chunks.add(chunk_key)
                meta = r['metadata']
                label = f"[Source: {meta.get('company','?')} | {meta.get('doc_name','?')} | Type: {stmt_type or 'auto'}]"
                all_evidence.append(f"{label}\n{r['evidence_text'][:1500]}")

    return "\n\n".join(all_evidence) if all_evidence else "No evidence found."


print(' Cell 11 done — Multi-hop retrieval ready')



✅ Cell 11 done — Multi-hop retrieval ready


In [20]:
# =====================================================================
# CELL 12: AGENT POST-PROCESSING
# Extract clean answers from verbose agent reasoning
# =====================================================================

def extract_agent_answer(agent_output: str, category: str) -> str:
    """
    Extract the clean final answer from agent's step-by-step reasoning.
    Prioritizes FINAL ANSWER marker, then falls back by category.
    """
    if not agent_output:
        return ""

    # ── Primary: look for FINAL ANSWER marker ────────────────────────
    # Capture everything after "FINAL ANSWER:" until end of line or paragraph
    final_match = re.search(
        r'FINAL ANSWER\s*[:\-]\s*(.+?)(?:\n\n|\Z)',
        agent_output, re.IGNORECASE | re.DOTALL
    )
    if final_match:
        answer = final_match.group(1).strip()
        # Clean up any trailing step markers
        answer = re.split(r'\n(?:STEP|Now |Begin)', answer)[0].strip()
        if category == 'numerical':
            return extract_answer_from_verbose(answer) or answer
        elif category == 'yes_no':
            # Extract just Yes/No + explanation, max 200 chars
            return answer[:200].strip()
        return answer[:300].strip()

    # ── Fallback: Numerical ──────────────────────────────────────────
    if category == 'numerical':
        # Find last "= [number]" pattern which is likely the calculation result
        all_calcs = re.findall(
            r'=\s*([\$\-]?[\d,\.]+\s*(?:billion|million|B|M|K|%)?)\s*(?:\n|$)',
            agent_output, re.IGNORECASE
        )
        if all_calcs:
            return all_calcs[-1].strip()  # last calculation = final result
        return extract_answer_from_verbose(agent_output)

    # ── Fallback: Yes/No ─────────────────────────────────────────────
    elif category == 'yes_no':
        # Find the LAST Yes or No in the text (most likely the conclusion)
        all_yn = list(re.finditer(r'\b(Yes|No)\b', agent_output, re.IGNORECASE))
        if all_yn:
            last = all_yn[-1]
            # Return from that Yes/No to end of its sentence
            snippet = agent_output[last.start():last.start()+250].strip()
            return snippet
        return agent_output[:200]

    # ── Fallback: Descriptive ────────────────────────────────────────
    else:
        lines = [l.strip() for l in agent_output.split('\n') if l.strip()]
        if lines:
            return '\n'.join(lines[-3:]).strip()
        return agent_output[-300:].strip()


def verify_numerical_answer(answer: str, evidence: str) -> tuple:
    """
    Returns (is_grounded, verification_message)
    """
    result = verify_tool(answer, evidence)
    is_grounded = result.startswith('VERIFIED')
    return is_grounded, result


print('Cell 12 done — Agent post-processing ready')

✅ Cell 12 done — Agent post-processing ready


In [21]:
# =====================================================================
# CELL 13: MAIN AGENT LOOP
# The full agentic evaluation replacing the original Cell 9b
# =====================================================================

from sentence_transformers import SentenceTransformer as SentTransformer
from sklearn.metrics.pairwise import cosine_similarity as cos_sim

sem_model = SentTransformer('all-MiniLM-L6-v2')

print(f'\n{"=" * 70}')
print(f'Llama 3.1 8B + AGENT (ReAct + Multi-Hop + Calculator) — {len(qa)} Questions')
print(f'{"=" * 70}')

correct_agent = 0
results_agent = []

for i, ex in enumerate(qa):
    q = ex['question']
    gold = ex['answer']
    fb_id = ex['financebench_id']
    evidence_list = ex.get('evidence', [])
    category = ex['category']

    # ── STEP 1: Multi-hop retrieval ───────────────────────────────────
    evidence_text = multi_hop_retrieve(q, k_per_hop=3)

    # ── STEP 2: Build category-specific agent prompt ──────────────────
    if category == 'numerical':
        prompt = build_agent_prompt_numerical(q, evidence_text)
    elif category == 'yes_no':
        prompt = build_agent_prompt_yesno(q, evidence_text)
    else:
        prompt = build_agent_prompt_descriptive(q, evidence_text)

    # ── STEP 3: Generate agent reasoning ─────────────────────────────
    try:
        agent_output = generate_answer(prompt, max_new_tokens=256)
    except Exception as e:
        print(f'   Generation error Q{i+1}: {e}')
        agent_output = 'ERROR'

    # ── STEP 4: Extract clean answer ──────────────────────────────────
    pred = extract_agent_answer(agent_output, category)

    # ── STEP 5: For numerical — run calculator on LLM's own formula ───
    calculator_used = False
    if category == 'numerical':
        # Strategy 1: Find "Calculation: A / B = result" and re-verify with our calculator
        calc_line = re.search(
            r'[Cc]alculation\s*[:\-]\s*([\d,\.\s\+\-\*\/\(\)]+)\s*=',
            agent_output
        )
        if calc_line:
            raw_formula = calc_line.group(1).strip().replace(',', '')
            calc_result = calculator_tool(raw_formula)
            if not calc_result.startswith('ERROR') and calc_result.strip():
                calculator_used = True
                pred = calc_result

        # Strategy 2: Find "From evidence: X = [num], Y = [num]" and build formula
        # Only if no FINAL ANSWER was found or pred looks wrong
        if not calculator_used:
            formula_line = re.search(
                r'[Ff]ormula\s*[:\-]\s*([^\n]+)',
                agent_output
            )
            if formula_line:
                raw = formula_line.group(1).strip()
                # Replace word numbers with numeric values already in pred
                nums_in_output = re.findall(r'[\d,]+(?:\.\d+)?', raw.replace(',',''))
                if len(nums_in_output) >= 2:
                    expr = re.sub(r'[^0-9\+\-\*\/\(\)\.\s]', ' ', raw)
                    expr = re.sub(r'\s+', '', expr)
                    calc_result = calculator_tool(expr)
                    if not calc_result.startswith('ERROR'):
                        calculator_used = True
                        pred = calc_result

    # ── STEP 6: Verify numerical answer against evidence ──────────────
    verified = False
    if category == 'numerical' and pred:
        is_grounded, _ = verify_numerical_answer(pred, evidence_text)
        verified = is_grounded

    # ── STEP 7: If not verified, retry with different statement type ──
    retry_done = False
    if category == 'numerical' and not verified:
        current_stmt = detect_statement_type(q)
        alt_stmts = [s for s in ['income', 'balance', 'cash_flow'] if s != current_stmt]
        for alt_stmt in alt_stmts:
            alt_evidence = retrieve_tool(q, statement_type=alt_stmt, k=3)
            alt_prompt = build_agent_prompt_numerical(q, alt_evidence)
            try:
                alt_output = generate_answer(alt_prompt, max_new_tokens=256)
                alt_pred = extract_agent_answer(alt_output, 'numerical')

                # Try calculator on alt output
                alt_calc_line = re.search(
                    r'[Cc]alculation\s*[:\-]\s*([\d,\.\s\+\-\*\/\(\)]+)\s*=',
                    alt_output
                )
                if alt_calc_line:
                    calc = calculator_tool(alt_calc_line.group(1).strip().replace(',',''))
                    if not calc.startswith('ERROR'):
                        alt_pred = calc

                is_grounded, _ = verify_numerical_answer(alt_pred, alt_evidence)
                if is_grounded:
                    pred = alt_pred
                    agent_output = alt_output
                    verified = True
                    retry_done = True
                    break
            except:
                pass

    # ── STEP 8: Correctness ───────────────────────────────────────────
    ok = is_correct(pred, gold)
    correct_agent += int(ok)

    results_agent.append({
        'financebench_id': fb_id,
        'company': ex['company'],
        'doc_name': ex['doc_name'],
        'question': q,
        'gold_answer': gold,
        'pred_raw': agent_output,
        'pred_answer': pred,
        'correct': ok,
        'category': category,
        'calculator_used': calculator_used,
        'verified': verified,
        'retry_done': retry_done,
    })

    if (i + 1) % 10 == 0:
        acc = correct_agent / (i + 1) * 100
        calc_used = sum(1 for r in results_agent if r['calculator_used'])
        print(f'  [{i+1:>3}/{len(qa)}] Accuracy: {acc:.1f}% | Calculator used: {calc_used}')

    # Show first 3 examples per category
    if i < 9:
        marker = '✅' if ok else '❌'
        print(f'\n  [{category.upper()}] {marker}')
        print(f'    Q:    {q[:70]}...')
        print(f'    Gold: {gold[:70]}')
        print(f'    Pred: {pred[:70]}')
        if calculator_used:
            print(f'    [Calculator used]')
        if retry_done:
            print(f'    [Retry retrieval done]')

print(f'\n  Overall Accuracy: {correct_agent}/{len(qa)} ({correct_agent/len(qa)*100:.1f}%)')

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Llama 3.1 8B + AGENT (ReAct + Multi-Hop + Calculator) — 150 Questions

  [NUMERICAL] ✅
    Q:    What is the FY2018 capital expenditure amount (in USD millions) for 3M...
    Gold: $1577.00
    Pred: $1,577M

  [NUMERICAL] ❌
    Q:    Assume that you are a public equities analyst. Answer the following qu...
    Gold: $8.70
    Pred: $8,738M

  [YES_NO] ❌
    Q:    Is 3M a capital-intensive business based on FY2022 data?...
    Gold: No, the company is managing its CAPEX and Fixed Assets pretty efficien
    Pred: Yes
3M is a capital-intensive business with a capital intensity ratio 

  [DESCRIPTIVE] ✅
    Q:    What drove operating margin change as of FY2022 for 3M? If operating m...
    Gold: Operating Margin for 3M in FY2022 has decreased by 1.7% primarily due 
    Pred: The increase in SG&A expenses as a percentage of sales, primarily due 

  [DESCRIPTIVE] ❌
    Q:    If we exclude the impact of M&A, which segment has dragged down 3M's o...
    Gold: The consumer segment shrunk by 0

In [22]:
# =====================================================================
# CELL 14: SPLIT EVALUATION METRICS
# =====================================================================

print(f'\n{"=" * 70}')
print(f'  Llama 3.1 8B + AGENT — SPLIT RESULTS')
print(f'{"=" * 70}')

# NUMERICAL
num_r = [r for r in results_agent if r['category'] == 'numerical']
num_acc_list    = [numerical_accuracy(r['pred_answer'], r['gold_answer']) for r in num_r]
num_hal_list    = [numerical_hallucination_rate(r['pred_answer'], evidence_map.get(r['financebench_id'], [])) for r in num_r]
num_cit_list    = [has_citation(r['pred_raw']) for r in num_r]
num_calc_count  = sum(1 for r in num_r if r['calculator_used'])
num_retry_count = sum(1 for r in num_r if r['retry_done'])
num_verified    = sum(1 for r in num_r if r['verified'])

print(f'\n   NUMERICAL ({len(num_r)} questions)')
print(f'  {"─" * 50}')
print(f'  Numerical Accuracy:      {np.mean(num_acc_list)*100:.1f}%  ({int(sum(num_acc_list))}/{len(num_r)})')
print(f'  Numerical Hallucination: {np.mean(num_hal_list)*100:.1f}%')
print(f'  Citation Rate:           {np.mean(num_cit_list)*100:.1f}%')
print(f'  Calculator Used:         {num_calc_count}/{len(num_r)} questions')
print(f'  Retry Retrieval:         {num_retry_count}/{len(num_r)} questions')
print(f'  Answers Verified:        {num_verified}/{len(num_r)} questions')

# YES/NO
yn_r = [r for r in results_agent if r['category'] == 'yes_no']
yn_correct = 0
yn_bleu_list = []
yn_sem_list  = []
yn_cit_list  = []
for r in yn_r:
    g = r['gold_answer'].strip().lower()
    p = r['pred_answer'].strip().lower()
    if (g.startswith('yes') and p.startswith('yes')) or (g.startswith('no') and p.startswith('no')):
        yn_correct += 1
    yn_bleu_list.append(calculate_bleu(r['gold_answer'], r['pred_answer']))
    emb = sem_model.encode([r['gold_answer'], r['pred_answer']])
    yn_sem_list.append(float(cos_sim([emb[0]], [emb[1]])[0][0]))
    yn_cit_list.append(has_citation(r['pred_raw']))

print(f'\n  YES/NO ({len(yn_r)} questions)')
print(f'  {"─" * 50}')
print(f'  Yes/No Accuracy:         {yn_correct/len(yn_r)*100:.1f}%  ({yn_correct}/{len(yn_r)})')
print(f'  BLEU Score:              {np.mean(yn_bleu_list):.3f}')
print(f'  Semantic Similarity:     {np.mean(yn_sem_list)*100:.1f}%')
print(f'  Citation Rate:           {np.mean(yn_cit_list)*100:.1f}%')

# DESCRIPTIVE
desc_r = [r for r in results_agent if r['category'] == 'descriptive']
desc_bleu_list   = []
desc_sem_list    = []
desc_keynum_list = []
desc_cit_list    = []
for r in desc_r:
    desc_bleu_list.append(calculate_bleu(r['gold_answer'], r['pred_answer']))
    emb = sem_model.encode([r['gold_answer'], r['pred_answer']])
    desc_sem_list.append(float(cos_sim([emb[0]], [emb[1]])[0][0]))
    gold_nums = extract_all_numbers(r['gold_answer'])
    pred_nums = extract_all_numbers(r['pred_answer'])
    if gold_nums:
        matched = 0
        for g in gold_nums:
            for p in pred_nums:
                if g != 0 and abs(p - g) / abs(g) < 0.05:
                    matched += 1; break
                elif g == 0 and abs(p) < 1e-6:
                    matched += 1; break
        desc_keynum_list.append(matched / len(gold_nums))
    else:
        desc_keynum_list.append(1.0)
    desc_cit_list.append(has_citation(r['pred_raw']))

print(f'\n  📝 DESCRIPTIVE ({len(desc_r)} questions)')
print(f'  {"─" * 50}')
print(f'  BLEU Score:              {np.mean(desc_bleu_list):.3f}')
print(f'  Semantic Similarity:     {np.mean(desc_sem_list)*100:.1f}%')
print(f'  Key Number Capture:      {np.mean(desc_keynum_list)*100:.1f}%')
print(f'  Citation Rate:           {np.mean(desc_cit_list)*100:.1f}%')

print(f'\n{"=" * 70}')
print(f'  OVERALL: {correct_agent}/{len(qa)} ({correct_agent/len(qa)*100:.1f}%)')
print(f'{"=" * 70}')


  Llama 3.1 8B + AGENT — SPLIT RESULTS

  📊 NUMERICAL (53 questions)
  ──────────────────────────────────────────────────
  Numerical Accuracy:      41.5%  (22/53)
  Numerical Hallucination: 45.3%
  Citation Rate:           100.0%
  Calculator Used:         19/53 questions
  Retry Retrieval:         4/53 questions
  Answers Verified:        31/53 questions

  ✅❌ YES/NO (37 questions)
  ──────────────────────────────────────────────────
  Yes/No Accuracy:         56.8%  (21/37)
  BLEU Score:              0.040
  Semantic Similarity:     60.1%
  Citation Rate:           21.6%

  📝 DESCRIPTIVE (60 questions)
  ──────────────────────────────────────────────────
  BLEU Score:              0.078
  Semantic Similarity:     62.1%
  Key Number Capture:      70.7%
  Citation Rate:           96.7%

  OVERALL: 47/150 (31.3%)


In [23]:
# =====================================================================
# CELL 15: SAVE RESULTS
# =====================================================================

import csv

output_path = 'results/agent_results.csv'
fieldnames = [
    'financebench_id', 'company', 'doc_name', 'question',
    'gold_answer', 'pred_answer', 'correct', 'category',
    'calculator_used', 'verified', 'retry_done'
]

with open(output_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    for r in results_agent:
        writer.writerow({k: r.get(k, '') for k in fieldnames})

print(f' Results saved to {output_path}')
print(f'\n AGENT EVALUATION COMPLETE!')
print(f'   Overall Accuracy: {correct_agent/len(qa)*100:.1f}%')


✅ Results saved to results/agent_results.csv

✅ AGENT EVALUATION COMPLETE!
   Overall Accuracy: 31.3%
